# 05 — Entraînement et validation temporelle

**Projet** : Prévision de consommation électrique multi-horizons avec scikit-learn
**Modèle** : `hist_gradient_boosting` (Gradient boosting par histogrammes (HistGradientBoostingRegressor))
**Métrique de décision** : `mape` (sens : minimize, seuil déclaré
4.0)

Entraîner un modèle de prévision est la partie facile. Ce qui distingue un projet exploitable d'un
prototype, ce sont quatre vérifications que ce notebook exécute **chiffres à l'appui** :

1. la validation est-elle **chronologique** ? Un protocole aléatoire sur une série temporelle est
   optimiste, et on mesure ici de combien ;
2. la **perte** optimisée est-elle celle que le métier pilote ? `squared_error` optimise la RMSE,
   alors que la décision se prend au MAPE ;
3. l'**écart d'apprentissage** est-il maîtrisé ? Un modèle qui mémorise le bruit AR(1) de la série
   tient en backtest et s'effondre en exploitation ;
4. les **artefacts** persistés suffisent-ils à l'inférence ? La chaîne de publication casse
   silencieusement s'il en manque un.

## Objectifs pédagogiques

1. Entraîner avec l'objet de production (`Trainer`), callbacks compris, sans toucher aux artefacts du pipeline.
1. Mesurer l'**optimisme** d'une validation croisée aléatoire face à des replis chronologiques.
1. Arbitrer la **perte d'entraînement** (`squared_error` contre `absolute_error`) sur quatre critères, pas un.
1. Piloter l'**écart train/validation** par la complexité des arbres.
1. Vérifier que chaque artefact persisté a un rôle précis dans la chaîne d'inférence, et faire un aller-retour de publication.
1. Chiffrer le **coût de ré-entraînement** et tester un déclencheur de dérive.

**Objectifs transverses du dépôt**

- Construire un jeu supervisé par expansion temporelle (origine x horizon) et formaliser le contrat d'antériorité de chaque feature : connue à l'origine, connue par avance, ou interdite.
- Comprendre pourquoi un split chronologique s'impose et ce que coûte concrètement une validation croisée aléatoire sur une série temporelle.
- Comparer un modèle appris à trois références triviales (persistance, naif saisonnier, moyenne glissante) et quantifier la valeur ajoutée réelle plutôt que le R².

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous
# les lignes INFO de production. Les erreurs réelles restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (4800 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 4800

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 4.0)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
# --- Contrat de prévision : tout est lu dans la configuration, rien n'est codé en dur -----------
FORECAST_CONF = dict(CONFIG.model_dump().get("load_forecasting") or {})
HORIZON_COLUMN = str(FORECAST_CONF.get("horizon_column") or "horizon_days")
HORIZONS = tuple(int(value) for value in (FORECAST_CONF.get("horizons") or ()))
LONG_HORIZON = int(FORECAST_CONF.get("long_horizon") or (max(HORIZONS) if HORIZONS else 7))
INTERVAL_LEVEL = float(FORECAST_CONF.get("interval_level") or 0.90)
INTERVAL_METHOD = str(FORECAST_CONF.get("interval_method") or "normalized_conformal")
INTERVAL_SCALE = str(FORECAST_CONF.get("interval_scale_column") or "load_last_observed")
BACKTEST_FOLDS = int(FORECAST_CONF.get("backtest_folds") or 5)

TARGET = str(CONFIG.data.target)
TIME_COLUMN = str(CONFIG.data.time_column or "origin_date")
TARGET_DATE = "target_date" if "target_date" in raw.columns else TIME_COLUMN
EVENT_COLUMN = "event_type" if "event_type" in raw.columns else None
NAIVE_COLUMN = "load_seasonal_naive" if "load_seasonal_naive" in raw.columns else None
PERSIST_COLUMN = "load_last_observed" if "load_last_observed" in raw.columns else None
TEMP_FORECAST = "temperature_forecast_c" if "temperature_forecast_c" in raw.columns else None


def mape(truth: Any, predicted: Any) -> float:
    """Mean absolute percentage error, in percent, ignoring zero denominators.

    Args:
        truth: Observed values.
        predicted: Forecast values.

    Returns:
        The MAPE in percent (``nan`` when nothing is measurable).
    """
    observed = np.asarray(truth, dtype="float64")
    forecast = np.asarray(predicted, dtype="float64")
    usable = np.isfinite(observed) & np.isfinite(forecast) & (np.abs(observed) > 1e-8)
    if not usable.any():
        return float("nan")
    return float(np.mean(np.abs((observed[usable] - forecast[usable]) / observed[usable])) * 100.0)


def mase(truth: Any, predicted: Any, reference: Any) -> float:
    """Mean absolute scaled error: model error over the naive reference error.

    Args:
        truth: Observed values.
        predicted: Forecast values.
        reference: Naive reference forecast on the same rows.

    Returns:
        The MASE (below 1 means better than the reference).
    """
    observed = np.asarray(truth, dtype="float64")
    forecast = np.asarray(predicted, dtype="float64")
    naive = np.asarray(reference, dtype="float64")
    usable = np.isfinite(observed) & np.isfinite(forecast) & np.isfinite(naive)
    scale = float(np.mean(np.abs(observed[usable] - naive[usable])))
    if not usable.any() or scale < 1e-9:
        return float("nan")
    return float(np.mean(np.abs(observed[usable] - forecast[usable])) / scale)


def daily_series(frame: pd.DataFrame) -> pd.DataFrame:
    """Collapse the (origin, horizon) panel into one row per target day.

    Le panel contient plusieurs lignes par jour cible (une par horizon) qui portent **la même**
    consommation : la série quotidienne se reconstruit en dédupliquant sur la date cible.

    Args:
        frame: Raw panel.

    Returns:
        One row per target day, sorted chronologically.
    """
    unique = frame.drop_duplicates(subset=[TARGET_DATE]).copy()
    unique[TARGET_DATE] = pd.to_datetime(unique[TARGET_DATE])
    return unique.sort_values(TARGET_DATE).reset_index(drop=True)


print(
    f"contrat de prévision : horizons={HORIZONS} colonne='{HORIZON_COLUMN}' "
    f"intervalle={INTERVAL_METHOD} (niveau {INTERVAL_LEVEL:.0%}, échelle '{INTERVAL_SCALE}')"
)
print(
    f"cible='{TARGET}' origine='{TIME_COLUMN}' cible_date='{TARGET_DATE}' "
    f"régimes='{EVENT_COLUMN}' naif='{NAIVE_COLUMN}'"
)

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    # Le pré-traitement supprime la colonne de groupe (identifiant, non modélisable), or une
    # métrique de classement se calcule **par groupe** puis se moyenne. Elle doit donc voyager à
    # côté des matrices, exactement comme dans `TrainPipeline` et dans les fixtures de tests.
    # `None` pour toute tâche sans structure de groupe : le comportement des autres projets est
    # inchangé.
    group_column = getattr(config.data, "group_column", None)

    def groups_of(split: pd.DataFrame | None) -> Any:
        if split is None or not group_column or group_column not in split.columns:
            return None
        return split[group_column].to_numpy()

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "groups_train": groups_of(enriched["train"]),
        "groups_val": groups_of(enriched["val"]),
        "groups_test": groups_of(enriched["test"]),
        "feature_names": list(pipeline.feature_names_out),
        # Colonnes de la matrice **avant** pré-traitement (donc avant one-hot). Indispensables dès
        # qu'un notebook ré-applique le pipeline à un nouveau cadre : sélectionner les colonnes de
        # `X_train` (après one-hot) sur un cadre enrichi lève un KeyError sur les modalités.
        "frame_columns": list(X_train_frame.columns),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

## 1. Entraînement avec l'objet de production

Le `Trainer` est celui qu'utilise `make train` : mêmes callbacks, mêmes métriques, même politique
de seuil. La seule différence est le répertoire de destination, temporaire, pour ne pas écraser les
artefacts publiés.

In [ ]:
# Entraînement avec l'objet de production (Trainer), dans un répertoire temporaire.
# Un notebook ne doit jamais écraser les artefacts de `make train` : il les reproduit à côté.
import tempfile

from src.models import build_model
from src.training.callbacks import MetricHistoryCallback, MetricThresholdCallback
from src.training.trainer import Trainer, TrainingData
from src.utils.paths import ProjectPaths

SANDBOX = ProjectPaths(root=Path(tempfile.mkdtemp(prefix="notebook-05-")))
SANDBOX.ensure()

MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
history = MetricHistoryCallback()
threshold = MetricThresholdCallback(
    f"val_{CONFIG.metrics.primary}",
    float(CONFIG.metrics.min_primary),
    mode="max" if str(CONFIG.metrics.direction) == "maximize" else "min",
)
trainer = Trainer(
    MODEL,
    config=CONFIG.train.model_dump(),
    paths=SANDBOX,
    metric_names=CONFIG.metrics.all_metrics,
    task=CONFIG.metrics.task,
    callbacks=[history, threshold],
    min_primary_metric=CONFIG.metrics.min_primary,
    primary_metric=f"val_{CONFIG.metrics.primary}",
    primary_direction=str(CONFIG.metrics.direction),
)
outcome = trainer.train(
    TrainingData(
        X_train=PREPARED["X_train"],
        y_train=PREPARED["y_train"],
        X_val=PREPARED["X_val"],
        y_val=PREPARED["y_val"],
        feature_names=PREPARED["feature_names"],
        task=CONFIG.metrics.task,
    )
)

PRIMARY_KEY = f"val_{CONFIG.metrics.primary}"
print(f"métrique primaire : {PRIMARY_KEY} = {outcome.metrics.get(PRIMARY_KEY, float('nan')):.4f}")
print(f"seuil déclaré     : {CONFIG.metrics.min_primary} (sens : {CONFIG.metrics.direction})")
print(f"gate du callback  : {'satisfait' if threshold.satisfied else 'NON satisfait'}")
print(f"durée             : {outcome.duration_seconds:.2f} s")
print(f"artefacts écrits  : {sorted(outcome.artifacts)}")
print(
    f"bac à sable       : {SANDBOX.root.name} (les artefacts de `make train` ne sont pas touchés)"
)
print()
print("métriques de validation :")
pd.Series(
    {
        key: round(float(value), 4)
        for key, value in outcome.metrics.items()
        if isinstance(value, (int, float))
    },
    name="valeur",
).to_frame()

**Ce qu'il faut retenir**

- Les callbacks ne sont pas de la décoration : `MetricThresholdCallback` transforme le seuil de qualité du manifeste en **gate d'exécution**, donc un ré-entraînement qui dégrade le modèle est signalé au lieu d'être publié.
- `MetricHistoryCallback` accumule l'historique des métriques. Sur un estimateur sans époque il n'a qu'une ligne, mais c'est lui qui alimente les courbes d'apprentissage des stacks itératives : le même code sert les deux familles.
- Les métriques de validation sont préfixées `val_` et celles d'entraînement ne le sont pas : c'est ce qui permet de calculer l'écart d'apprentissage sans ambiguïté.

## 2. Validation chronologique contre validation aléatoire

La question n'est pas théorique : elle décide si les chiffres annoncés au métier sont crédibles.
On exécute les deux protocoles sur les **mêmes** données.

In [ ]:
# Validation chronologique contre validation aléatoire : la mesure de l'optimisme.
# Une validation croisée aléatoire sur une série temporelle mélange les époques : des lignes
# d'entraînement encadrent temporellement des lignes de test, ce qui est une fuite par recouvrement.
from sklearn.model_selection import KFold

from src.models import build_model

X_all = pd.concat([PREPARED["X_train"], PREPARED["X_val"]], ignore_index=True)
y_all = pd.concat(
    [
        pd.Series(np.asarray(PREPARED["y_train"], dtype="float64")),
        pd.Series(np.asarray(PREPARED["y_val"], dtype="float64")),
    ],
    ignore_index=True,
)
frame_all = pd.concat([PREPARED["splits"].train, PREPARED["splits"].val], ignore_index=True)

random_scores = []
for train_index, test_index in KFold(n_splits=5, shuffle=True, random_state=CONFIG.seed).split(
    X_all
):
    model = build_model(
        CONFIG, feature_names=PREPARED["feature_names"], params=dict(CONFIG.model.params)
    )
    model.fit(X_all.iloc[train_index], y_all.to_numpy()[train_index])
    forecast = np.asarray(model.predict(X_all.iloc[test_index]), dtype="float64").ravel()
    random_scores.append(mape(y_all.to_numpy()[test_index], forecast))

# Équivalent chronologique : cinq replis consécutifs, sans mélange
# (chaque repli s'entraîne sur tout ce qui précède son bloc test).
N_FOLDS = 5
edges = np.linspace(0, len(X_all), N_FOLDS + 2).astype(int)  # N replis -> N+2 bornes
temporal_scores = []
for index in range(N_FOLDS):
    train_index = np.arange(0, edges[index + 1])
    test_index = np.arange(edges[index + 1], edges[index + 2])
    if len(test_index) == 0:
        continue
    model = build_model(
        CONFIG, feature_names=PREPARED["feature_names"], params=dict(CONFIG.model.params)
    )
    model.fit(X_all.iloc[train_index], y_all.to_numpy()[train_index])
    forecast = np.asarray(model.predict(X_all.iloc[test_index]), dtype="float64").ravel()
    temporal_scores.append(mape(y_all.to_numpy()[test_index], forecast))

comparison = pd.DataFrame(
    {
        "protocole": ["KFold aléatoire", "replis chronologiques"],
        "MAPE moyen %": [
            round(float(np.mean(random_scores)), 3),
            round(float(np.mean(temporal_scores)), 3),
        ],
        "écart-type": [
            round(float(np.std(random_scores)), 3),
            round(float(np.std(temporal_scores)), 3),
        ],
        "replis": [len(random_scores), len(temporal_scores)],
    }
)
print(comparison.to_string(index=False))
optimism = float(comparison["MAPE moyen %"].iloc[0] - comparison["MAPE moyen %"].iloc[1])
print(f"\noptimisme du protocole aléatoire : {optimism:+.3f} point de MAPE")
print("périodes couvertes par chaque repli chronologique :")
for index in range(5):
    block = frame_all.iloc[edges[index + 1] : edges[index + 2]]
    if len(block):
        print(
            f"  repli {index + 1} : {pd.to_datetime(block[TIME_COLUMN]).min().date()} -> "
            f"{pd.to_datetime(block[TIME_COLUMN]).max().date()} ({len(block)} lignes)"
        )

**Ce qu'il faut retenir**

- Le protocole aléatoire est systématiquement **optimiste** : chaque repli contient des lignes dont les voisines temporelles immédiates sont dans l'entraînement. Sur une série à forte autocorrélation, cela revient à évaluer une interpolation, pas une prévision.
- L'écart mesuré est le prix de l'erreur de protocole. Il se reporte directement sur la décision : un modèle qui « passe » le seuil en KFold aléatoire peut le manquer en replis chronologiques.
- Les replis chronologiques couvrent des périodes **différentes** (donc des régimes différents) : leur dispersion n'est pas du bruit, c'est de l'information sur la stabilité saisonnière du modèle.
- C'est la raison pour laquelle `conf/model/default.yaml` désactive la validation croisée aléatoire : le backtest à origine glissante de l'évaluateur la remplace.

## 3. La perte d'entraînement face à la métrique de décision

Le métier pilote au MAPE. L'estimateur, lui, optimise une perte. Les aligner n'est pas
automatiquement le bon choix — voici les deux côtés du compromis.

In [ ]:
# Perte d'entraînement : le métier pilote au MAPE, l'estimateur optimise une perte.
# `squared_error` optimise la RMSE ; `absolute_error` est alignée sur le MAPE mais produit des
# gradients discontinus. La comparaison est le seul moyen honnête de trancher.
from src.models import build_model

rows = []
for loss in ("squared_error", "absolute_error"):
    params = {**dict(CONFIG.model.params), "loss": loss}
    try:
        model = build_model(CONFIG, feature_names=PREPARED["feature_names"], params=params)
        model.fit(PREPARED["X_train"], PREPARED["y_train"])
        forecast = np.asarray(model.predict(PREPARED["X_test"]), dtype="float64").ravel()
        truth = np.asarray(PREPARED["y_test"], dtype="float64")
        rows.append(
            {
                "perte": loss,
                "MAPE test %": round(mape(truth, forecast), 3),
                "MAE MW": round(float(np.mean(np.abs(truth - forecast))), 1),
                "RMSE MW": round(float(np.sqrt(np.mean((truth - forecast) ** 2))), 1),
                "biais %": round(float(np.mean((forecast - truth) / truth)) * 100.0, 2),
                "P95 erreur %": round(
                    float(np.percentile(np.abs((truth - forecast) / truth)) * 100.0), 2
                ),
            }
        )
    except Exception as error:
        rows.append({"perte": loss, "MAPE test %": None, "erreur": str(error)[:70]})

loss_table = pd.DataFrame(rows)
print(loss_table.to_string(index=False))

**Ce qu'il faut retenir**

- `absolute_error` aligne la perte sur le MAPE, mais ses gradients sont discontinus autour de zéro : l'optimisation est moins stable et l'ensemble converge plus lentement.
- `squared_error` pénalise davantage les grosses erreurs, ce qui est **souhaitable** ici : une pointe hivernale ratée coûte plus cher qu'une erreur moyenne en été. La RMSE plus élevée n'est pas un défaut, c'est la trace de cette priorité.
- Le biais est le critère à surveiller : une perte absolue tend à prévoir la médiane, donc à sous-prévoir les pointes — exactement ce qu'un acheteur d'énergie redoute.
- Le choix configuré (`squared_error`) est donc un arbitrage documenté, pas un réglage par défaut subi.

## 4. Écart d'apprentissage : ce que la complexité achète et ce qu'elle coûte

In [ ]:
# Écart d'apprentissage : train / validation / test, et son pilotage par la complexité.
from src.models import build_model

truth_train = np.asarray(PREPARED["y_train"], dtype="float64")
truth_val = np.asarray(PREPARED["y_val"], dtype="float64")
truth_test = np.asarray(PREPARED["y_test"], dtype="float64")

rows = []
for leaves in (7, 15, 31, 63, 127):
    params = {**dict(CONFIG.model.params), "max_leaf_nodes": leaves}
    model = build_model(CONFIG, feature_names=PREPARED["feature_names"], params=params)
    model.fit(PREPARED["X_train"], PREPARED["y_train"])
    scores = {
        "train": mape(truth_train, np.asarray(model.predict(PREPARED["X_train"]), dtype="float64")),
        "val": mape(truth_val, np.asarray(model.predict(PREPARED["X_val"]), dtype="float64")),
        "test": mape(truth_test, np.asarray(model.predict(PREPARED["X_test"]), dtype="float64")),
    }
    rows.append(
        {
            "feuilles": leaves,
            "MAPE train %": round(scores["train"], 3),
            "MAPE val %": round(scores["val"], 3),
            "MAPE test %": round(scores["test"], 3),
            "écart train/val": round(scores["val"] - scores["train"], 3),
            "rapport val/train": round(scores["val"] / scores["train"], 2)
            if scores["train"] > 1e-9
            else None,
        }
    )

overfit_table = pd.DataFrame(rows)
print(overfit_table.to_string(index=False))
print(f"\nconfiguration retenue : max_leaf_nodes={CONFIG.model.params.get('max_leaf_nodes')}")

fig, axis = plt.subplots(figsize=(8, 4.2))
axis.plot(overfit_table["feuilles"], overfit_table["MAPE train %"], marker="o", label="train")
axis.plot(overfit_table["feuilles"], overfit_table["MAPE val %"], marker="s", label="validation")
axis.plot(overfit_table["feuilles"], overfit_table["MAPE test %"], marker="^", label="test")
axis.set_xscale("log", base=2)
axis.set_xlabel("nombre maximal de feuilles par arbre")
axis.set_ylabel("MAPE (%)")
axis.set_title("L'écart d'apprentissage se lit avant le score")
axis.legend()
axis.grid(alpha=0.3)
plt.show()

**Ce qu'il faut retenir**

- Quand le nombre de feuilles augmente, le MAPE d'entraînement chute et celui de validation cesse de suivre : la divergence **est** le surapprentissage. Le point retenu est le dernier avant la divergence, pas le meilleur score de validation.
- Le rapport val/train est plus parlant que l'écart absolu : un modèle à 1,7 % en entraînement et 3,5 % en validation (rapport ≈ 2) est sain sur une série bruitée ; un rapport proche de 1 signale une validation trop proche de l'entraînement, donc probablement mal découpée.
- Le test suit la validation de près, ce qui est le signal recherché : si le test s'écartait nettement, la période de test contiendrait un régime absent de la validation — c'est précisément ce que le notebook 06 vérifie sur les intervalles.

## 5. Artefacts persistés et aller-retour d'inférence

In [ ]:
# Ce que l'entraînement persiste, et ce dont l'inférence a besoin.
# La chaîne de publication casse silencieusement si un seul de ces objets manque ou est désaligné.
artifacts_root = PATHS.artifacts_dir
rows = []
for relative, purpose in [
    ("models/model.joblib", "modèle entraîné (prédiction ponctuelle)"),
    (
        "models/preprocessing.joblib",
        "pré-traitement ajusté sur train (mêmes colonnes, mêmes échelles)",
    ),
    ("models/feature_builder.joblib", "recettes de features apprises sur train"),
    (
        "models/resolved_config.json",
        "configuration résolue : horizons, méthode d'intervalle, seuils",
    ),
    ("models/model_card.json", "carte du modèle : identité, métriques, empreinte des données"),
    (
        "reports/intervals.csv",
        "calibration des intervalles par horizon (méthode + score + échelle)",
    ),
    ("reports/per_horizon.csv", "erreur attendue par horizon, publiée avec chaque prévision"),
]:
    path = artifacts_root / relative
    rows.append(
        {
            "artefact": relative,
            "présent": path.exists(),
            "taille": f"{path.stat().st_size / 1024:.1f} ko" if path.exists() else "-",
            "rôle": purpose,
        }
    )
artifacts_table = pd.DataFrame(rows)
print(artifacts_table.to_string(index=False))
print()

# Aller-retour d'inférence : le prédicteur reconstruit-il la même prévision que le modèle nu ?
from src.inference.predictor import Predictor  # noqa: E402

try:
    predictor = Predictor.from_config(CONFIG.model_dump(), PATHS)
    sample = predictor.sample_inputs(n=4) if hasattr(predictor, "sample_inputs") else None
    if sample is not None:
        published = predictor.predict(sample)
        columns = [
            column
            for column in [
                predictor.horizon_column,
                f"{TARGET}_prediction",
                "lower_mw",
                "upper_mw",
                "expected_mape_pct",
                "confidence",
                "interval_method",
            ]
            if column in published.columns
        ]
        print("publication du prédicteur (4 enregistrements) :")
        print(published[columns].to_string(index=False))
except Exception as error:
    print(f"prédicteur indisponible (lancez `make train` puis `make evaluate`) : {error}")

**Ce qu'il faut retenir**

- Quatre objets sont nécessaires pour publier : le modèle, le pré-traitement, le constructeur de features et la **configuration résolue**. Sans ce dernier, l'évaluateur et le prédicteur devraient deviner la colonne d'horizon ou la méthode d'intervalle.
- Les tables de calibration (`intervals.csv`, `per_horizon.csv`) font partie du contrat d'inférence : le prédicteur y lit l'intervalle et l'erreur attendue, donc une évaluation absente dégrade la publication au lieu de la rendre fausse.
- L'aller-retour prouve l'alignement : le prédicteur reconstruit les mêmes features que l'entraînement, dans le même ordre. Un désalignement de colonnes est l'incident le plus fréquent en mise en production, et il est silencieux.

## 6. Coût de ré-entraînement et déclencheur de dérive

In [ ]:
# Coût et déclencheur de ré-entraînement : la question que pose l'astreinte, pas le notebook.
import time

from src.models import build_model

started = time.perf_counter()
model = build_model(
    CONFIG, feature_names=PREPARED["feature_names"], params=dict(CONFIG.model.params)
)
model.fit(PREPARED["X_train"], PREPARED["y_train"])
fit_seconds = time.perf_counter() - started
forecast = np.asarray(model.predict(PREPARED["X_test"]), dtype="float64").ravel()
score_now = mape(PREPARED["y_test"], forecast)

started = time.perf_counter()
model.predict(PREPARED["X_test"].head(1))
predict_seconds = time.perf_counter() - started

print(f"entraînement ({len(PREPARED['X_train'])} lignes) : {fit_seconds:.2f} s")
print(f"prédiction unitaire                        : {predict_seconds * 1000:.1f} ms")
print(f"MAPE de référence sur le test              : {score_now:.3f} %")
print()

# Déclencheur de dérive : on compare l'erreur des derniers jours à celle du reste de la période.
# Un test statistique simple vaut mieux qu'un seuil arbitraire sur le MAPE, parce qu'il tient
# compte du volume : 50 lignes ne prouvent rien, 500 si.
ordered = PREPARED["splits"].test.sort_values(TIME_COLUMN).reset_index(drop=True)
X_ordered = PREPARED["pipeline"].transform(
    PREPARED["builder"].transform(ordered).loc[:, PREPARED["frame_columns"]]
)
truth_ordered = ordered[TARGET].to_numpy(dtype="float64")
forecast_ordered = np.asarray(model.predict(X_ordered), dtype="float64").ravel()
errors = 100.0 * np.abs((truth_ordered - forecast_ordered) / truth_ordered)

window = max(len(errors) // 5, 20)
recent, past = errors[-window:], errors[:-window]
welch = (recent.mean() - past.mean()) / np.sqrt(
    recent.var(ddof=1) / len(recent) + past.var(ddof=1) / len(past)
)
print(
    f"fenêtre récente : {window} lignes "
    f"({pd.to_datetime(ordered[TIME_COLUMN]).iloc[-window:].min().date()} -> "
    f"{pd.to_datetime(ordered[TIME_COLUMN]).max().date()})"
)
print(f"  erreur moyenne récente : {recent.mean():.2f} %")
print(f"  erreur moyenne passée  : {past.mean():.2f} %")
print(f"  statistique de Welch   : {welch:+.2f} (seuil de décision ±1,96)")
print(
    f"  conclusion             : "
    f"{'ré-entraîner' if welch > 1.96 else 'aucune dérive significative'}"
)

**Ce qu'il faut retenir**

- Le coût d'entraînement se compte en secondes sur ce volume : la vraie contrainte n'est pas le calcul mais la **revue** qui accompagne chaque nouveau modèle (validation des seuils, comparaison au précédent, journalisation).
- Le déclencheur de dérive compare la fenêtre récente au reste de la période avec un test de Welch : il tient compte du volume, donc il ne déclenche pas sur 30 lignes bruitées.
- Une dérive confirmée appelle un ré-entraînement ; une dérive **saisonnière** récurrente appelle autre chose — une variable de régime ou un recalibrage. Distinguer les deux est l'objet du notebook 06.